# Gold Layer: Sales & Orders Analytics

This notebook implements the Sales and Orders Analytics marts in the Databricks Lakehouse:
- Pre-aggregates `order_items` and `payments` to `order_id` grain to prevent Cartesian join fan-out.
- Materializes `workspace.ecompulse_gold.daily_sales_analytics` with daily order counts, revenue, AOV, payment breakdowns, running total revenue, 7-day rolling moving average, and DoD growth rates with division-by-zero guards.
- Materializes `workspace.ecompulse_gold.monthly_sales_analytics` with monthly order counts, monthly revenue, AOV, payment breakdowns, MoM growth rates, and running totals.
- Persists all datasets to Delta Lake tables in Unity Catalog.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
orders = spark.read.table("workspace.ecompulse_silver.orders")
payments = spark.read.table("workspace.ecompulse_silver.payments")
order_items = spark.read.table("workspace.ecompulse_silver.order_items")

In [0]:
# Pre-aggregate order_items to order_id grain to eliminate fan-out
order_items_agg = order_items.groupBy("order_id").agg(
    F.round(F.sum("price"), 2).alias("total_item_price"),
    F.round(F.sum("freight_value"), 2).alias("total_freight_value"),
    F.count("order_item_id").alias("total_items_count")
)

In [0]:
# Pre-aggregate payments to order_id grain to eliminate fan-out
payments_agg = payments.groupBy("order_id").agg(
    F.round(F.sum("payment_value"), 2).alias("total_payment_value"),
    F.round(F.sum(F.when(F.col("payment_type") == "credit_card", F.col("payment_value")).otherwise(0.0)), 2).alias("credit_card_payment_value"),
    F.round(F.sum(F.when(F.col("payment_type") == "boleto", F.col("payment_value")).otherwise(0.0)), 2).alias("boleto_payment_value"),
    F.round(F.sum(F.when(F.col("payment_type") == "voucher", F.col("payment_value")).otherwise(0.0)), 2).alias("voucher_payment_value"),
    F.round(F.sum(F.when(F.col("payment_type") == "debit_card", F.col("payment_value")).otherwise(0.0)), 2).alias("debit_card_payment_value"),
    F.count("payment_sequential").alias("payment_installments_count")
)

In [0]:
# Join orders with pre-aggregated items and payments (1:1 join at order_id grain)
order_sales = orders.join(order_items_agg, on="order_id", how="left") \
                    .join(payments_agg, on="order_id", how="left") \
                    .fillna({
                        "total_item_price": 0.0,
                        "total_freight_value": 0.0,
                        "total_items_count": 0,
                        "total_payment_value": 0.0,
                        "credit_card_payment_value": 0.0,
                        "boleto_payment_value": 0.0,
                        "voucher_payment_value": 0.0,
                        "debit_card_payment_value": 0.0,
                        "payment_installments_count": 0
                    })

In [0]:
# Compute daily sales aggregation
daily_base = order_sales.withColumn(
    "order_date", F.to_date(F.col("order_purchase_timestamp"))
).groupBy("order_date").agg(
    F.countDistinct("order_id").alias("order_count"),
    F.round(F.sum("total_payment_value"), 2).alias("revenue"),
    F.round(F.sum("total_freight_value"), 2).alias("total_freight"),
    F.round(F.sum("credit_card_payment_value"), 2).alias("credit_card_revenue"),
    F.round(F.sum("boleto_payment_value"), 2).alias("boleto_revenue"),
    F.round(F.sum("voucher_payment_value"), 2).alias("voucher_revenue"),
    F.round(F.sum("debit_card_payment_value"), 2).alias("debit_card_revenue")
).withColumn(
    "total_orders", F.col("order_count")
).withColumn(
    "average_order_value",
    F.when(F.col("order_count") > 0, F.round(F.col("revenue") / F.col("order_count"), 2)).otherwise(0.0)
)

# Apply PySpark window functions for daily metrics
daily_window = Window.orderBy("order_date")
running_total_window = daily_window.rowsBetween(Window.unboundedPreceding, Window.currentRow)
moving_avg_window = daily_window.rowsBetween(-6, Window.currentRow)

daily_sales_analytics = daily_base.withColumn(
    "running_total_revenue",
    F.round(F.sum("revenue").over(running_total_window), 2)
).withColumn(
    "moving_avg_7d_revenue",
    F.round(F.avg("revenue").over(moving_avg_window), 2)
).withColumn(
    "prev_day_revenue",
    F.lag("revenue", 1).over(daily_window)
).withColumn(
    "dod_growth_pct",
    F.when(
        (F.col("prev_day_revenue").isNotNull()) & (F.col("prev_day_revenue") != 0),
        F.round(((F.col("revenue") - F.col("prev_day_revenue")) / F.col("prev_day_revenue")) * 100, 2)
    ).otherwise(None)
)

In [0]:
# Save daily sales to Delta Lake table
daily_sales_analytics.write.mode("overwrite") \
    .format("delta") \
    .saveAsTable("workspace.ecompulse_gold.daily_sales_analytics")

In [0]:
# Compute monthly sales aggregation from daily mart
monthly_base = daily_sales_analytics.withColumn(
    "year_month", F.date_format(F.date_trunc("month", F.col("order_date")), "yyyy-MM")
).groupBy("year_month").agg(
    F.sum("order_count").alias("order_count"),
    F.round(F.sum("revenue"), 2).alias("monthly_revenue"),
    F.round(F.sum("total_freight"), 2).alias("total_freight"),
    F.round(F.sum("credit_card_revenue"), 2).alias("credit_card_revenue"),
    F.round(F.sum("boleto_revenue"), 2).alias("boleto_revenue"),
    F.round(F.sum("voucher_revenue"), 2).alias("voucher_revenue"),
    F.round(F.sum("debit_card_revenue"), 2).alias("debit_card_revenue")
).withColumn(
    "total_orders", F.col("order_count")
).withColumn(
    "revenue", F.col("monthly_revenue")
).withColumn(
    "average_order_value",
    F.when(F.col("order_count") > 0, F.round(F.col("monthly_revenue") / F.col("order_count"), 2)).otherwise(0.0)
)

# Apply PySpark window functions for monthly metrics
monthly_window = Window.orderBy("year_month")
monthly_running_window = monthly_window.rowsBetween(Window.unboundedPreceding, Window.currentRow)

monthly_sales_analytics = monthly_base.withColumn(
    "prev_month_revenue",
    F.lag("monthly_revenue", 1).over(monthly_window)
).withColumn(
    "mom_growth_rate",
    F.when(
        (F.col("prev_month_revenue").isNotNull()) & (F.col("prev_month_revenue") != 0),
        F.round(((F.col("monthly_revenue") - F.col("prev_month_revenue")) / F.col("prev_month_revenue")) * 100, 2)
    ).otherwise(None)
).withColumn(
    "running_total_revenue",
    F.round(F.sum("monthly_revenue").over(monthly_running_window), 2)
)

In [0]:
# Save monthly sales to Delta Lake table
monthly_sales_analytics.write.mode("overwrite") \
    .format("delta") \
    .saveAsTable("workspace.ecompulse_gold.monthly_sales_analytics")

In [0]:
%sql
SELECT * FROM workspace.ecompulse_gold.daily_sales_analytics ORDER BY order_date DESC LIMIT 10;

In [0]:
%sql
SELECT * FROM workspace.ecompulse_gold.monthly_sales_analytics ORDER BY year_month DESC LIMIT 10;